[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-02-data-shapes.ipynb#scrollTo=10a2b3c4)

---
# Day 2 · Data Shapes & Sources
**certified-journeys / altair-certified** · Day 2 · Tidy Data

> **Goal for today:** Understand why Altair requires tidy (long-form) data, reshape wide data using pandas `melt` and Altair's `fold` transform, load data from a URL, and handle missing values before plotting.


In [ ]:
%pip install -q altair vega-datasets


## Step 1 · Load a dataset and inspect its shape

Before plotting, it's essential to understand the structure of your data.
We'll load the `gapminder` dataset, which tracks life expectancy, population,
and GDP per capita for many countries over decades.

Key inspection checklist:
- `.shape` — rows × columns
- `.dtypes` — data types per column
- `.isnull().sum()` — missing value counts
- `.head()` — first few rows


In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data

# Load the gapminder dataset
gapminder = data.gapminder()

print("Shape:", gapminder.shape)
print()
print("Dtypes:")
print(gapminder.dtypes)
print()
print("Missing values:")
print(gapminder.isnull().sum())
print()
gapminder.head(6)


### What just happened?
- The gapminder dataset is **already in long form** — each row is one country-year observation.
- Columns like `life_expect`, `pop`, and `fertility` are each a separate variable — this is **tidy data**.
- **No missing values** here, which is ideal. Real datasets often need cleaning before plotting.
- Understanding dtypes is critical: Altair will infer `:Q` vs `:N` from dtypes if you use them without suffixes.


## Step 2 · Long-form vs Wide-form data

**Tidy / Long-form data** (what Altair expects):
- One row per observation
- One column per variable
- Easy to map columns to encoding channels

**Wide-form data** (common in spreadsheets):
- One row per entity, multiple measurement columns
- Requires reshaping before Altair can use it

Example comparison:

| Format | Country | 2000 | 2005 | 2010 |
|--------|---------|------|------|------|
| Wide   | USA     | 76.5 | 77.3 | 78.5 |

| Format | Country | Year | Life Expectancy |
|--------|---------|------|----------------|
| Long   | USA     | 2000 | 76.5 |
| Long   | USA     | 2005 | 77.3 |
| Long   | USA     | 2010 | 78.5 |


In [ ]:
# Demonstrate the problem: create a WIDE dataset artificially
# Pivot gapminder to wide form (years become columns)
gm_subset = gapminder[gapminder['country'].isin(['China', 'India', 'United States'])]

wide_df = gm_subset.pivot_table(
    index='country',
    columns='year',
    values='life_expect'
).reset_index()

print("Wide-form data (years as columns):")
print(wide_df.to_string())
print()
print("Shape:", wide_df.shape)


### What just happened?
- We pivoted the data so each year becomes its own column — typical spreadsheet layout.
- **This is hard for Altair to use**: there's no single `year` column to map to the x-axis.
- To plot life expectancy over time, Altair needs a `year` column and a `life_expect` column — not 9 separate year columns.
- The next step shows how to reshape this back to long form.


## Step 3 · Reshape wide→long with `pandas.melt`

`pd.melt()` is the standard pandas tool for converting wide-form to long-form:

```python
pd.melt(
    df,
    id_vars=['country'],    # columns to keep as-is (row identifiers)
    var_name='year',        # new column name for the old column headers
    value_name='life_expect' # new column name for the cell values
)
```


In [ ]:
# Reshape wide→long with melt
long_df = pd.melt(
    wide_df,
    id_vars=['country'],      # keep country as identifier
    var_name='year',           # year becomes a column
    value_name='life_expect'   # values become one column
)

# Convert year to integer for proper axis ordering
long_df['year'] = long_df['year'].astype(int)

print("Long-form data after melt:")
print(long_df.head(9).to_string())
print("\nShape:", long_df.shape)


### What just happened?
- `melt` **unpivoted** the year columns into rows — each country-year pair is now one row.
- **Now Altair can use it directly**: map `year` to x, `life_expect` to y, `country` to color.
- The shape went from 3×10 (wide) to 27×3 (long) — fewer columns, more rows.
- Converting `year` to `int` ensures proper numeric ordering on the axis.


In [ ]:
# Now plot the reshaped data — works perfectly with long form
melted_chart = alt.Chart(long_df).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),             # Ordinal so years are evenly spaced
    y=alt.Y('life_expect:Q', title='Life Expectancy (years)',
             scale=alt.Scale(zero=False)),         # Don't force y-axis to start at 0
    color=alt.Color('country:N', title='Country'),
    tooltip=['country:N', 'year:O', 'life_expect:Q']
).properties(
    title='Life Expectancy Over Time (reshaped from wide-form)',
    width=480,
    height=280
)

melted_chart


### What just happened?
- After reshaping, the chart took only 10 lines — **the melt was the hard part**.
- `alt.Scale(zero=False)` prevents Altair from forcing the y-axis to start at 0,
  making the trend differences more visible.
- `'year:O'` (Ordinal) spaces years evenly; use `'year:Q'` (Quantitative) for proportional spacing.
- **All three lines share the same x-axis column** — only possible because data is in long form.


## Step 4 · Use the `fold` transform (reshape inside Vega-Lite)

Altair's `fold` transform reshapes wide data **inside the chart spec** — no pandas needed.
This is useful when you want to keep your data manipulation in one place.

```python
alt.Chart(wide_df).transform_fold(
    fold=['col_a', 'col_b', 'col_c'],  # columns to fold
    as_=['variable', 'value']           # new column names
)
```

After `transform_fold`, the folded columns become rows, and you can encode
`'variable:N'` and `'value:Q'` directly.


In [ ]:
# Use fold transform to plot multiple metrics from the cars dataset
# without reshaping in pandas
cars = data.cars()

# We want to compare Horsepower and Miles_per_Gallon distributions
# across origins — using fold to avoid reshaping
fold_chart = alt.Chart(cars).transform_fold(
    fold=['Horsepower', 'Miles_per_Gallon'],
    as_=['Metric', 'Value']
).mark_boxplot().encode(
    x=alt.X('Origin:N', title='Country of Origin'),
    y=alt.Y('Value:Q', title='Value'),
    color='Origin:N',
    column=alt.Column('Metric:N', title='')  # facet by metric
).properties(
    title='Horsepower and MPG distributions by Origin (fold transform)',
    width=180,
    height=250
)

fold_chart


### What just happened?
- `transform_fold` converted two columns into rows **inside the Vega-Lite spec** — the pandas
  DataFrame was never modified.
- `column=alt.Column('Metric:N')` creates a **faceted chart** — one panel per metric.
- `mark_boxplot()` is a composite mark that shows median, quartiles, and outliers automatically.
- **When to use fold vs melt**: use `transform_fold` for quick in-spec reshaping; use `melt` when
  you need the reshaped data for other purposes too (filtering, grouping in pandas first).


## Step 5 · Load data from a URL directly

Altair charts can reference data **by URL** — the browser fetches it at render time.
This is ideal for large datasets or datasets hosted externally.

```python
alt.Chart('https://example.com/data.csv').mark_point()...
# or
alt.Chart(alt.UrlData(url='...', format=alt.DataFormat(type='csv')))
```

Advantages of URL loading:
- No Python data wrangling for the initial load
- The chart spec is smaller (no data embedded)
- Works in static HTML exports


In [ ]:
# Load data directly from a URL (vega-datasets CDN)
# The 'stocks' dataset is available at the Vega CDN
stocks_url = 'https://cdn.jsdelivr.net/npm/vega-datasets@latest/data/stocks.csv'

url_chart = alt.Chart(stocks_url).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('price:Q', title='Stock Price (USD)'),
    color=alt.Color('symbol:N', title='Company'),
    tooltip=['symbol:N', 'date:T', 'price:Q']
).properties(
    title='Tech Stock Prices Over Time (loaded from URL)',
    width=520,
    height=300
)

url_chart


### What just happened?
- We passed a **URL string** directly to `alt.Chart()` — no `pd.read_csv()` needed.
- Altair embedded the URL in the Vega-Lite spec; the browser fetches the CSV when rendering.
- The column names (`date`, `price`, `symbol`) come from the CSV headers — we just reference them as strings.
- **Limitation**: URL data is fetched client-side, so it won't work in offline environments. For offline use, load with pandas first.


## Step 6 · Handle missing values and type coercions

Real data has missing values. Altair silently skips `NaN` rows in most cases, but
you should handle them explicitly to avoid silent gaps in your charts.

Common pre-plotting cleaning steps:

| Issue | Pandas fix |
|-------|------------|
| Missing numerics | `df.dropna(subset=['col'])` or `df['col'].fillna(median)` |
| Wrong dtype | `df['col'].astype(float)` |
| Mixed types in column | `pd.to_numeric(df['col'], errors='coerce')` |
| Date strings | `pd.to_datetime(df['col'])` |


In [ ]:
# Demonstrate missing value handling with the movies dataset
movies = data.movies()

print("Missing values in movies dataset:")
print(movies.isnull().sum().sort_values(ascending=False).head(8))
print(f"\nTotal rows: {len(movies)}")


In [ ]:
# Clean the movies dataset before plotting
movies_clean = movies.dropna(subset=[
    'Rotten_Tomatoes_Rating',
    'IMDB_Rating',
    'Production_Budget'
]).copy()

# Coerce Production_Budget to numeric (some entries may be strings)
movies_clean['Production_Budget'] = pd.to_numeric(
    movies_clean['Production_Budget'], errors='coerce'
)

# Drop any rows where coercion produced NaN
movies_clean = movies_clean.dropna(subset=['Production_Budget'])

print(f"Rows after cleaning: {len(movies_clean)} (removed {len(movies) - len(movies_clean)})")

# Now plot safely
clean_chart = alt.Chart(movies_clean).mark_point(opacity=0.5).encode(
    x=alt.X('Rotten_Tomatoes_Rating:Q', title='Rotten Tomatoes (%)'),
    y=alt.Y('IMDB_Rating:Q', title='IMDB Rating'),
    size=alt.Size('Production_Budget:Q', title='Budget', scale=alt.Scale(range=[20, 400])),
    color=alt.Color('Major_Genre:N', title='Genre'),
    tooltip=['Title', 'Major_Genre', 'Rotten_Tomatoes_Rating', 'IMDB_Rating']
).properties(
    title='Movie Ratings: Rotten Tomatoes vs IMDB (cleaned data)',
    width=520,
    height=360
)

clean_chart


### What just happened?
- `dropna(subset=[...])` removed rows missing any of the columns we need — surgical, not a full drop.
- `pd.to_numeric(..., errors='coerce')` converts non-numeric strings to `NaN` instead of raising.
- `size=alt.Size(...)` adds a **fourth encoding channel**: bubble size represents production budget.
- `opacity=0.5` (mark-level, not encoded) reduces overplotting when many points overlap.
- **Always inspect missing values before plotting** — silent gaps in time series or wrong counts in bar charts are common symptoms of unhandled NaNs.


In [ ]:
# Challenge: Take the gapminder dataset (already loaded above)
# It has columns: country, year, cluster, pop, life_expect, fertility
#
# 1. Filter to only the year 2000
# 2. Use transform_fold to fold 'life_expect' and 'fertility' into long form
# 3. Build a faceted bar chart (one column per metric)
#    x = cluster:O, y = mean(Value:Q), color = cluster:O
#
# Expected: 2 side-by-side bar charts showing avg life_expect and fertility
# by world region cluster in 2000

gapminder = data.gapminder()

# Step 1: filter to year 2000
# gm2000 = gapminder[...]

# Step 2 + 3: transform_fold + faceted bar chart
# challenge = alt.Chart(gm2000).transform_fold(
#     fold=[...],
#     as_=['Metric', 'Value']
# ).mark_bar().encode(
#     x=...,
#     y=...,
#     color=...,
#     column=...
# ).properties(width=180, height=220)
# challenge


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Tidy / long-form data | One row per observation, one column per variable — what Altair needs |
| Wide-form data | Many columns per entity — must reshape before plotting |
| `pd.melt()` | pandas tool to unpivot wide→long; use for reusable long-form data |
| `transform_fold` | Altair in-spec fold; use for quick reshaping without touching the DataFrame |
| URL data sources | Pass a URL string to `alt.Chart()` — browser fetches at render time |
| Missing value handling | `dropna`, `fillna`, `pd.to_numeric(errors='coerce')` before plotting |
| `alt.Scale(zero=False)` | Prevents y-axis from forcing zero — use for trend visibility |

> **Tip:** Altair works best with tidy (long-form) data — one row per observation, one column per variable. Reshape with `melt` before plotting if your data is wide.

---
## What's next
**Day 3** → Marks Deep Dive — explore every major mark type (point, line, bar, area, rect, arc, text, rule) with real datasets and build a layered chart challenge.

Mark Day 2 complete in your [tracker](../index.html).
